# Phase 1 Metric Formula Contract Remediation Plan

Claim-closed orchestration notebook. This notebook plans remediation for the unresolved metric-formula contract after the post-formula-decision governance update. It does not choose a formula, change runtime semantics, alter thresholds, edit logits or metrics, rerun controls, or open claims.

Scientific-integrity rule: this notebook may only write a remediation plan. Any future contract update must be a separate reviewed code/config/docs change with tests before any controls are rerun.

In [ ]:
# Cell 1 - Bootstrap repo and run unit tests.

from pathlib import Path
import base64
import getpass
import hashlib
import json
import subprocess
from datetime import datetime, timezone

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'
RUN_UNITTESTS = True

def run(cmd, cwd=None, check=True):
    printable = ' '.join(str(part) for part in cmd)
    if cmd and cmd[0] == 'git':
        printable = printable.replace(str(REPO_URL), 'https://github.com/BrianNguyen29/eeg-ds004752.git')
    print('$', printable)
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)

if not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    run(['git', '-c', f'http.extraHeader=Authorization: Basic {auth}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    print('Repo exists:', REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

print('Repo:', REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR, check=False)

if RUN_UNITTESTS:
    run(['python', '-m', 'unittest', 'tests.unit.test_phase1_final_metric_formula_contract_remediation_plan'], cwd=REPO_DIR)


In [ ]:
# Cell 2 - Select reviewed source and keep launch disabled by default.

EXPECTED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = ARTIFACT_ROOT / 'prereg/20260418T161442014597Z/prereg_bundle.json'

# Use None to resolve latest.txt, or pin an already reviewed post-formula governance update run.
PINNED_POST_FORMULA_DECISION_GOVERNANCE_RUN = None

OUTPUT_ROOT = ARTIFACT_ROOT / 'phase1_final_metric_formula_contract_remediation_plan'
PLAN_ROOT = ARTIFACT_ROOT / 'phase1_final_metric_formula_contract_remediation_plan_manual_hold'

RUN_METRIC_FORMULA_CONTRACT_REMEDIATION_PLAN = False
REQUIRED_ACK = 'I reviewed the metric formula contract remediation plan and understand it selects no formula changes no runtime code reruns controls or claims'
MANUAL_LAUNCH_ACK = ''
FIX_MARKER = 'phase1_final_metric_formula_contract_remediation_plan_v1_20260423'
print('Notebook fix marker:', FIX_MARKER)


def resolve_latest(root: Path, required_file: str) -> Path:
    latest = root / 'latest.txt'
    if latest.exists():
        candidate = Path(latest.read_text().strip())
        if (candidate / required_file).exists():
            return candidate
    runs = sorted(path for path in root.iterdir() if path.is_dir()) if root.exists() else []
    for candidate in reversed(runs):
        if (candidate / required_file).exists():
            return candidate
    raise FileNotFoundError(f'No run containing {required_file} under {root}')

POST_FORMULA_DECISION_GOVERNANCE_RUN = Path(PINNED_POST_FORMULA_DECISION_GOVERNANCE_RUN) if PINNED_POST_FORMULA_DECISION_GOVERNANCE_RUN else resolve_latest(
    ARTIFACT_ROOT / 'phase1_final_post_formula_decision_governance_update',
    'phase1_final_post_formula_decision_governance_update_summary.json',
)

print('Post-formula-decision governance run:', POST_FORMULA_DECISION_GOVERNANCE_RUN)
print('Run flag:', RUN_METRIC_FORMULA_CONTRACT_REMEDIATION_PLAN)


In [ ]:
# Cell 3 - Validate prereg and post-formula governance boundary.

def read_json(path: Path):
    return json.loads(path.read_text())

def sha256(path: Path):
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_sha256 = sha256(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

post_summary = read_json(POST_FORMULA_DECISION_GOVERNANCE_RUN / 'phase1_final_post_formula_decision_governance_update_summary.json')
metric_status = read_json(POST_FORMULA_DECISION_GOVERNANCE_RUN / 'phase1_final_metric_formula_contract_status.json')
post_claim_state = read_json(POST_FORMULA_DECISION_GOVERNANCE_RUN / 'phase1_final_post_formula_decision_governance_claim_state.json')

assert post_summary.get('status') == 'phase1_final_post_formula_decision_governance_update_recorded'
assert post_summary.get('claim_ready') is False
assert post_summary.get('headline_phase1_claim_open') is False
assert post_summary.get('claims_opened') is False
assert post_summary.get('formula_decision') == 'unresolved'
assert post_summary.get('selected_formula') is None
assert post_summary.get('metric_formula_contract_claim_evaluable') is False
assert metric_status.get('next_step') == 'do_not_rerun_controls_until_metric_formula_contract_is_resolved'
assert 'metric_formula_contract:metric_formula_contract_unresolved' in post_claim_state.get('blockers', [])

print(json.dumps({
    'post_formula_governance_run': str(POST_FORMULA_DECISION_GOVERNANCE_RUN),
    'formula_decision': post_summary.get('formula_decision'),
    'selected_formula': post_summary.get('selected_formula'),
    'metric_formula_next_step': post_summary.get('metric_formula_next_step'),
    'claims_opened': post_summary.get('claims_opened'),
}, indent=2))


In [ ]:
# Cell 4 - Record manual-hold plan and command preview.

PLAN_ROOT.mkdir(parents=True, exist_ok=True)
plan_dir = PLAN_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
plan_dir.mkdir(parents=True, exist_ok=False)

runner_available = (REPO_DIR / 'src/phase1/final_metric_formula_contract_remediation_plan.py').exists()
preflight_blockers = []
if not runner_available:
    preflight_blockers.append('src/phase1/final_metric_formula_contract_remediation_plan.py missing')

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_metric_formula_contract_remediation_plan',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--post-formula-decision-governance-run', str(POST_FORMULA_DECISION_GOVERNANCE_RUN),
    '--output-root', str(OUTPUT_ROOT),
]

plan = {
    'status': 'phase1_final_metric_formula_contract_remediation_plan_manual_hold',
    'created_utc': plan_dir.name,
    'runner_available': runner_available,
    'run_flag': RUN_METRIC_FORMULA_CONTRACT_REMEDIATION_PLAN,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'post_formula_decision_governance_run': str(POST_FORMULA_DECISION_GOVERNANCE_RUN),
    'formula_decision': post_summary.get('formula_decision'),
    'preflight_blockers': preflight_blockers,
    'command': cmd,
    'scientific_integrity_rule': 'This plan drafts remediation scope only; it selects no formula and reruns nothing.',
}
(plan_dir / 'phase1_final_metric_formula_contract_remediation_plan_manual_hold.json').write_text(json.dumps(plan, indent=2) + '\n')
print(json.dumps(plan, indent=2))


In [ ]:
# Cell 5 - Manual launch gate.

if preflight_blockers:
    raise RuntimeError(f'Preflight blockers must be resolved before remediation planning: {preflight_blockers}')
elif not RUN_METRIC_FORMULA_CONTRACT_REMEDIATION_PLAN:
    print('HELD: Metric-formula contract remediation plan was not launched because manual flag is False.')
    print('NEXT: review the source artifacts, then rerun only with explicit claim-closed acknowledgement if appropriate.')
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not record remediation plan without explicit claim-closed acknowledgement.')
else:
    run(cmd, cwd=REPO_DIR)
    print('Metric-formula contract remediation plan completed. Review generated artifacts before any future contract proposal.')


In [ ]:
# Cell 6 - Inspect latest output if launched.

latest_output = None
if RUN_METRIC_FORMULA_CONTRACT_REMEDIATION_PLAN and MANUAL_LAUNCH_ACK == REQUIRED_ACK:
    latest_output = resolve_latest(OUTPUT_ROOT, 'phase1_final_metric_formula_contract_remediation_plan_summary.json')
    summary = read_json(latest_output / 'phase1_final_metric_formula_contract_remediation_plan_summary.json')
    scope = read_json(latest_output / 'phase1_final_metric_formula_contract_remediation_scope.json')
    claim_state = read_json(latest_output / 'phase1_final_metric_formula_contract_remediation_claim_state.json')

    print('Latest metric-formula remediation plan output:', latest_output)
    print('Formula decision:', summary.get('formula_decision'))
    print('Selected formula:', summary.get('selected_formula'))
    print('Remediation scope:', summary.get('remediation_scope'))
    print('Code change allowed now:', summary.get('code_change_allowed_now'))
    print('Runtime formula change allowed now:', summary.get('runtime_formula_change_allowed_now'))
    print('Controls rerun allowed now:', summary.get('controls_rerun_allowed_now'))
    print('Threshold change allowed now:', summary.get('threshold_change_allowed_now'))
    print('Claims opened:', summary.get('claims_opened'))
    print('Next step:', summary.get('next_step'))
    print('Claim blockers:', claim_state.get('blockers'))
else:
    print('No remediation-plan output inspected because launch is held.')


In [ ]:
# Cell 7 - Assertions and review note.

if latest_output is not None:
    assert summary.get('status') == 'phase1_final_metric_formula_contract_remediation_plan_recorded', summary.get('claim_blockers')
    assert summary.get('claim_ready') is False
    assert summary.get('headline_phase1_claim_open') is False
    assert summary.get('claims_opened') is False
    assert summary.get('formula_decision') == 'unresolved'
    assert summary.get('selected_formula') is None
    assert summary.get('code_change_allowed_now') is False
    assert summary.get('runtime_formula_change_allowed_now') is False
    assert summary.get('controls_rerun_allowed_now') is False
    assert summary.get('threshold_change_allowed_now') is False
    assert summary.get('claim_opening_allowed_now') is False
    assert scope.get('remediation_scope') == 'metric_formula_contract_docs_config_planning_only'
    assert 'metric_formula_contract_unresolved' in claim_state.get('blockers', [])

    review_note = {
        'status': 'phase1_final_metric_formula_contract_remediation_plan_review_pass_claim_closed',
        'reviewed_run': str(latest_output),
        'formula_decision': summary.get('formula_decision'),
        'selected_formula': summary.get('selected_formula'),
        'checks_passed': [
            'remediation_plan_recorded',
            'claim_ready_false',
            'headline_claim_open_false',
            'formula_not_selected',
            'code_change_not_allowed_now',
            'runtime_formula_change_not_allowed_now',
            'controls_rerun_not_allowed_now',
            'threshold_change_not_allowed_now',
            'metric_formula_unresolved_blocker_preserved',
        ],
        'next_step': summary.get('next_step'),
        'not_ok_to_claim': claim_state.get('not_ok_to_claim'),
    }
    note_path = latest_output / 'phase1_final_metric_formula_contract_remediation_plan_review_note.json'
    note_path.write_text(json.dumps(review_note, indent=2) + '\n')
    print('Review note written:', note_path)
    print(json.dumps(review_note, indent=2))
else:
    print('Assertions skipped because launch is held.')


In [ ]:
# Cell 8 - Closeout message.

print('================ Phase 1 Metric Formula Contract Remediation Plan Closeout ================')
print('Notebook fix marker:', FIX_MARKER)
print('Plan source of truth:', plan_dir)
print('Runner available:', runner_available)
print('Run flag:', RUN_METRIC_FORMULA_CONTRACT_REMEDIATION_PLAN)
print('Post-formula-decision governance run:', POST_FORMULA_DECISION_GOVERNANCE_RUN)
print('Formula decision:', post_summary.get('formula_decision'))
print('Expected locked prereg identity hash:', EXPECTED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)

if latest_output is None:
    print('HELD: Metric-formula contract remediation plan was not launched because manual flag is False.')
    print('NEXT: rerun only with explicit claim-closed acknowledgement if the reviewed sources are correct.')
else:
    print('Latest metric-formula remediation plan output:', latest_output)
    print('Selected formula:', summary.get('selected_formula'))
    print('Code change allowed now:', summary.get('code_change_allowed_now'))
    print('Controls rerun allowed now:', summary.get('controls_rerun_allowed_now'))
    print('Next step:', summary.get('next_step'))
    print('CHECK REQUIRED: Review phase1_final_metric_formula_contract_remediation_decision_memo.md before drafting any contract/code/config change.')

print('NOT OK TO CLAIM: This notebook does not prove decoder efficacy, A2d efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')
